In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from skimage.feature import graycomatrix, graycoprops

# ── Konfigurasi ──────────────────────────────────────────────
BASE_DIR = Path("grape")

TRAIN_DIR = BASE_DIR / "train"
VALID_DIR = BASE_DIR / "valid"

TARGET_SIZE = 256 # Ukuran target untuk gambar (misalnya 128, 256, atau 32)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Ambil nama kelas dari folder train
CLASS_NAMES = sorted([
    d.name for d in TRAIN_DIR.iterdir()
    if d.is_dir()
])

print("Kelas:", CLASS_NAMES)
print("Jumlah kelas:", len(CLASS_NAMES))

Kelas: ['Grape Esca (Black_Measles)', 'Grape Leaf blight (Isariopsis_Leaf_Spot)', 'grape leaf Healthy', 'grape leaf black rot']
Jumlah kelas: 4


# Preprocessing

In [59]:
def segment_leaf_hsv(image_bgr, min_area_ratio=0.01):
    """
    Segmentasi daun dengan HSV + pembersihan komponen.
    - Menggabungkan rentang warna daun (hijau/kuning/coklat/merah)
    - Menolak piksel low-saturation (background kusam)
    - Menyisakan komponen terbesar agar background tidak ikut
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    # Rentang warna daun (lebih ketat)
    lower_green = np.array([20, 35, 25])
    upper_green = np.array([95, 255, 255])

    lower_yellow = np.array([15, 40, 35])
    upper_yellow = np.array([35, 255, 255])

    lower_brown = np.array([5, 50, 20])
    upper_brown = np.array([20, 255, 220])

    lower_red1 = np.array([0, 50, 30])
    upper_red1 = np.array([12, 255, 255])
    lower_red2 = np.array([160, 50, 30])
    upper_red2 = np.array([180, 255, 255])

    # Mask warna
    mask_color  = cv2.inRange(hsv, lower_green, upper_green)
    mask_color |= cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask_color |= cv2.inRange(hsv, lower_brown, upper_brown)
    mask_color |= cv2.inRange(hsv, lower_red1, upper_red1)
    mask_color |= cv2.inRange(hsv, lower_red2, upper_red2)

    # Tolak background abu/gelap (S dan V terlalu rendah)
    mask_sv = cv2.inRange(hsv, np.array([0, 35, 25]), np.array([180, 255, 255]))
    mask = cv2.bitwise_and(mask_color, mask_sv)

    # Morphological cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Keep komponen terbesar (leaf utama)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    refined = np.zeros_like(mask)

    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]  # abaikan background index 0
        largest_idx = 1 + np.argmax(areas)
        largest_area = stats[largest_idx, cv2.CC_STAT_AREA]

        h, w = mask.shape[:2]
        min_area = int(min_area_ratio * h * w)

        if largest_area >= min_area:
            refined[labels == largest_idx] = 255
        else:
            refined = mask.copy()
    else:
        refined = mask.copy()

    # Haluskan tepi akhir
    refined = cv2.medianBlur(refined, 5)
    result = cv2.bitwise_and(image_bgr, image_bgr, mask=refined)
    return result, refined

In [60]:
def apply_clahe(image_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    """
    Menerapkan CLAHE pada kanal L (Lightness) di ruang warna LAB.
    Kontras diperbaiki secara lokal tanpa mengubah pigmen warna asli.
    """
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_eq = clahe.apply(l_channel)

    lab_eq = cv2.merge([l_eq, a_channel, b_channel])
    result = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)
    return result

In [ ]:
def zero_pad_and_resize(image_bgr, target_size):
    """
    1. Zero-padding → jadikan persegi (sisi = max(h, w))
    2. Resize       → target_size × target_size
    Aspect ratio asli dipertahankan; area tambahan diisi hitam.
    """
    h, w = image_bgr.shape[:2]
    max_side = max(h, w)

    # Buat canvas hitam persegi
    canvas = np.zeros((max_side, max_side, 3), dtype=np.uint8)

    # Letakkan gambar di tengah canvas
    y_offset = (max_side - h) // 2
    x_offset = (max_side - w) // 2
    canvas[y_offset:y_offset + h, x_offset:x_offset + w] = image_bgr

    # Resize ke target
    resized = cv2.resize(canvas, (target_size, target_size),
                         interpolation=cv2.INTER_AREA)
    return resized

In [62]:
def normalize_minmax(image_bgr):
    """
    Min-Max Normalization: pixel / 255.0
    Output bertipe float32 dengan rentang [0, 1].
    """
    return image_bgr.astype(np.float32) / 255.0

In [63]:
def augment_image(image):
    """
    Augmentasi geometris acak:
      - Rotasi 90°/180°/270°
      - Flip horizontal
      - Flip vertikal
      - Flip horizontal + vertikal
    Mengembalikan list gambar hasil augmentasi (TIDAK termasuk gambar asli).
    """
    augmented = []

    # Rotasi 90°, 180°, 270°
    for k in [1, 2, 3]:
        augmented.append(np.rot90(image, k=k).copy())

    # Flip horizontal
    augmented.append(np.flip(image, axis=1).copy())

    # Flip vertikal
    augmented.append(np.flip(image, axis=0).copy())

    # Flip horizontal + vertikal (setara rotasi 180° + flip)
    augmented.append(np.flip(image, axis=(0, 1)).copy())

    return augmented

# Ekstraksi Fitur Warna dengan Histogram HSV

In [64]:
def extract_hsv_hist_features(
    image_bgr,
    grid_size=(2, 2),   # 4-kuadran
    bins_h=8,
    bins_s=8,
    bins_v=8
):
    """
    Ekstraksi fitur warna HSV histogram per kuadran.
    Output: vektor fitur 1D.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]
    gh, gw = grid_size

    features = []

    for i in range(gh):
        for j in range(gw):
            y0 = int(i * h / gh)
            y1 = int((i + 1) * h / gh)
            x0 = int(j * w / gw)
            x1 = int((j + 1) * w / gw)

            cell = hsv[y0:y1, x0:x1]

            hist_h = cv2.calcHist([cell], [0], None, [bins_h], [0, 180])
            hist_s = cv2.calcHist([cell], [1], None, [bins_s], [0, 256])
            hist_v = cv2.calcHist([cell], [2], None, [bins_v], [0, 256])

            hist_h = cv2.normalize(hist_h, None).flatten()
            hist_s = cv2.normalize(hist_s, None).flatten()
            hist_v = cv2.normalize(hist_v, None).flatten()

            features.extend(hist_h)
            features.extend(hist_s)
            features.extend(hist_v)

    return np.array(features, dtype=np.float32)

# Ekstraksi Fitur Tekstur dengan GLCM

In [65]:
def extract_glcm_features(
    image_bgr,
    distances=(1, 2, 3),
    angles=(0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    levels=32
):
    """
    Ekstraksi fitur tekstur menggunakan GLCM pada grayscale.
    Output: vektor fitur 1D.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Kuantisasi grayscale ke levels tertentu
    gray_q = np.floor(gray.astype(np.float32) * levels / 256.0).astype(np.uint8)
    gray_q = np.clip(gray_q, 0, levels - 1)

    glcm = graycomatrix(
        gray_q,
        distances=list(distances),
        angles=list(angles),
        levels=levels,
        symmetric=True,
        normed=True
    )

    props = ["contrast", "homogeneity", "energy", "correlation"]
    features = []

    for prop in props:
        values = graycoprops(glcm, prop)
        features.extend(values.flatten())

    return np.array(features, dtype=np.float32)

In [ ]:
def preprocess_for_features(image_bgr):
    """
    Menggunakan preprocessing yang sudah ada:
    1. HSV masking
    2. CLAHE
    3. Zero-padding + resize
    4. Normalisasi
    Lalu dikembalikan ke uint8 agar aman untuk histogram dan GLCM.
    """
    seg, _ = segment_leaf_hsv(image_bgr)
    clahe = apply_clahe(seg)
    resized = zero_pad_and_resize(clahe, TARGET_SIZE)
    normalized = normalize_minmax(resized)

    # Balik ke uint8 supaya cocok untuk ekstraksi fitur
    processed = (normalized * 255).astype(np.uint8)
    return processed

In [67]:
def extract_combined_features(image_bgr):
    hsv_features = extract_hsv_hist_features(image_bgr)
    glcm_features = extract_glcm_features(image_bgr)
    return np.concatenate([hsv_features, glcm_features]).astype(np.float32)

In [68]:
sample_class = CLASS_NAMES[0]
sample_dir = TRAIN_DIR / sample_class

sample_file = sorted([
    f for f in sample_dir.iterdir()
    if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
])[0]

image = cv2.imread(str(sample_file))

processed = preprocess_for_features(image)
features = extract_combined_features(processed)

print("File:", sample_file.name)
print("Shape original:", image.shape)
print("Shape processed:", processed.shape)
print("Jumlah fitur:", len(features))
print("5 fitur pertama:", features[:5])

File: Grape Esca (Black_Measles) (1).JPG
Shape original: (256, 256, 3)
Shape processed: (256, 256, 3)
Jumlah fitur: 144
5 fitur pertama: [0.96783555 0.2278103  0.1067413  0.         0.        ]


In [69]:
def generate_feature_names(
    grid_size=(2, 2),
    bins_h=8,
    bins_s=8,
    bins_v=8,
    distances=(1, 2, 3),
    angles=(0, np.pi/4, np.pi/2, 3*np.pi/4)
):

    feature_names = []

    # =====================================================
    # HSV HISTOGRAM
    # =====================================================

    gh, gw = grid_size

    quadrant = 1

    for i in range(gh):
        for j in range(gw):

            # H
            for b in range(bins_h):
                feature_names.append(
                    f"H_Q{quadrant}_B{b+1}"
                )

            # S
            for b in range(bins_s):
                feature_names.append(
                    f"S_Q{quadrant}_B{b+1}"
                )

            # V
            for b in range(bins_v):
                feature_names.append(
                    f"V_Q{quadrant}_B{b+1}"
                )

            quadrant += 1

    # =====================================================
    # GLCM
    # =====================================================

    props = [
        "contrast",
        "homogeneity",
        "energy",
        "correlation"
    ]

    angle_names = {
        0: "0",
        np.pi/4: "45",
        np.pi/2: "90",
        3*np.pi/4: "135"
    }

    for prop in props:

        for d in distances:

            for a in angles:

                feature_names.append(
                    f"{prop}_d{d}_a{angle_names[a]}"
                )

    return feature_names

In [70]:
feature_names = generate_feature_names()

print("Jumlah nama fitur:", len(feature_names))

print("\n10 nama fitur pertama:")
print(feature_names[:10])

Jumlah nama fitur: 144

10 nama fitur pertama:
['H_Q1_B1', 'H_Q1_B2', 'H_Q1_B3', 'H_Q1_B4', 'H_Q1_B5', 'H_Q1_B6', 'H_Q1_B7', 'H_Q1_B8', 'S_Q1_B1', 'S_Q1_B2']


In [71]:
train_rows = []

for class_name in CLASS_NAMES:

    class_dir = TRAIN_DIR / class_name

    files = sorted([
        f for f in class_dir.iterdir()
        if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    ])

    print(f"Processing train: {class_name}")

    for file in tqdm(files):

        image = cv2.imread(str(file))
        if image is None:
            continue

        processed = preprocess_for_features(image)
        features = extract_combined_features(processed)

        row = {
            "filename": file.name,
            "label": class_name
        }

        for name, val in zip(feature_names, features):
            row[name] = float(val)

        train_rows.append(row)

print("Total train:", len(train_rows))

train_df = pd.DataFrame(train_rows)

Processing train: Grape Esca (Black_Measles)


100%|██████████| 1920/1920 [00:13<00:00, 140.23it/s]


Processing train: Grape Leaf blight (Isariopsis_Leaf_Spot)


100%|██████████| 1722/1722 [00:11<00:00, 146.66it/s]


Processing train: grape leaf Healthy


100%|██████████| 1749/1749 [00:18<00:00, 95.10it/s] 


Processing train: grape leaf black rot


100%|██████████| 1944/1944 [00:15<00:00, 122.90it/s]


Total train: 7335


In [72]:
train_df

,filename,label,H_Q1_B1,H_Q1_B2,H_Q1_B3,H_Q1_B4,H_Q1_B5,H_Q1_B6,H_Q1_B7,H_Q1_B8,...,correlation_d1_a90,correlation_d1_a135,correlation_d2_a0,correlation_d2_a45,correlation_d2_a90,correlation_d2_a135,correlation_d3_a0,correlation_d3_a45,correlation_d3_a90,correlation_d3_a135
0,Grape Esca (Black_Measles) (1).JPG,Grape Esca (Black_Measles),0.967836,0.227810,0.106741,0.000000,0.000000,0.0,0.000000,0.001751,...,0.850585,0.823468,0.780187,0.817100,0.794187,0.823468,0.730173,0.744667,0.754219,0.750931
1,Grape Esca (Black_Measles) (10).JPG,Grape Esca (Black_Measles),0.940276,0.296857,0.166573,0.002322,0.000430,0.0,0.000172,0.002322,...,0.911382,0.882747,0.843140,0.872292,0.833673,0.882747,0.781331,0.776399,0.777354,0.790766
2,Grape Esca (Black_Measles) (100).JPG,Grape Esca (Black_Measles),0.991801,0.069146,0.107465,0.000214,0.000000,0.0,0.000000,0.000500,...,0.911387,0.887812,0.836524,0.862714,0.839640,0.887812,0.789775,0.789118,0.798009,0.822461
3,Grape Esca (Black_Measles) (1000).JPG,Grape Esca (Black_Measles),0.975616,0.173541,0.134321,0.000393,0.000000,0.0,0.000000,0.003851,...,0.828204,0.804306,0.794194,0.822199,0.782041,0.804306,0.761275,0.770965,0.746098,0.748921
4,Grape Esca (Black_Measles) (1001).JPG,Grape Esca (Black_Measles),0.976598,0.071421,0.202845,0.001456,0.000536,0.0,0.000077,0.002605,...,0.827986,0.821835,0.793476,0.803729,0.781317,0.821835,0.760602,0.747752,0.745300,0.770462
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7330,grape leaf black rot (995).JPG,grape leaf black rot,0.958624,0.277729,0.062081,0.000000,0.000000,0.0,0.000000,0.007252,...,0.866207,0.835100,0.813164,0.843430,0.803954,0.835100,0.764969,0.782186,0.761127,0.766630
7331,grape leaf black rot (996).JPG,grape leaf black rot,0.953588,0.215965,0.208166,0.000000,0.000000,0.0,0.000000,0.026396,...,0.867068,0.844316,0.813969,0.835939,0.805289,0.844316,0.766290,0.768020,0.762488,0.783644
7332,grape leaf black rot (997).JPG,grape leaf black rot,0.939671,0.050999,0.327081,0.086227,0.000000,0.0,0.000000,0.000343,...,0.904399,0.861476,0.810748,0.858208,0.845128,0.861476,0.757007,0.787309,0.808415,0.795165
7333,grape leaf black rot (998).JPG,grape leaf black rot,0.838820,0.032926,0.540781,0.053416,0.000000,0.0,0.000000,0.000000,...,0.887454,0.858769,0.843180,0.857081,0.827468,0.858769,0.809850,0.808078,0.795909,0.809586


In [73]:
train_df.to_csv(f"features/{TARGET_SIZE}/train_features.csv", index=False)

print("Berhasil disimpan!")

Berhasil disimpan!


In [74]:
valid_rows = []

for class_name in CLASS_NAMES:

    class_dir = VALID_DIR / class_name

    files = sorted([
        f for f in class_dir.iterdir()
        if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    ])

    print(f"Processing valid: {class_name}")

    for file in tqdm(files):

        image = cv2.imread(str(file))
        if image is None:
            continue

        processed = preprocess_for_features(image)
        features = extract_combined_features(processed)

        row = {
            "filename": file.name,
            "label": class_name
        }

        for name, val in zip(feature_names, features):
            row[name] = float(val)

        valid_rows.append(row)

print("Total valid:", len(valid_rows))

valid_df = pd.DataFrame(valid_rows)

Processing valid: Grape Esca (Black_Measles)


100%|██████████| 480/480 [00:03<00:00, 141.51it/s]


Processing valid: Grape Leaf blight (Isariopsis_Leaf_Spot)


100%|██████████| 430/430 [00:02<00:00, 144.34it/s]


Processing valid: grape leaf Healthy


100%|██████████| 435/435 [00:03<00:00, 123.79it/s]


Processing valid: grape leaf black rot


100%|██████████| 480/480 [00:03<00:00, 135.86it/s]

Total valid: 1825


In [75]:
valid_df

,filename,label,H_Q1_B1,H_Q1_B2,H_Q1_B3,H_Q1_B4,H_Q1_B5,H_Q1_B6,H_Q1_B7,H_Q1_B8,...,correlation_d1_a90,correlation_d1_a135,correlation_d2_a0,correlation_d2_a45,correlation_d2_a90,correlation_d2_a135,correlation_d3_a0,correlation_d3_a45,correlation_d3_a90,correlation_d3_a135
0,Grape Esca (Black_Measles) (1).JPG,Grape Esca (Black_Measles),0.968922,0.245989,0.022668,0.000000,0.000000,0.000000,0.000000,0.012899,...,0.913288,0.859918,0.794605,0.861198,0.835605,0.859918,0.726245,0.764012,0.779675,0.765202
1,Grape Esca (Black_Measles) (10).JPG,Grape Esca (Black_Measles),0.909495,0.065123,0.410158,0.018300,0.003694,0.000859,0.000000,0.000000,...,0.834995,0.794279,0.722153,0.774767,0.782196,0.794279,0.672027,0.695954,0.742993,0.724864
2,Grape Esca (Black_Measles) (100).JPG,Grape Esca (Black_Measles),0.958880,0.126726,0.253944,0.000000,0.000000,0.000000,0.000000,0.001719,...,0.846094,0.783299,0.726759,0.797954,0.793470,0.783299,0.682182,0.729625,0.756690,0.710173
3,Grape Esca (Black_Measles) (101).JPG,Grape Esca (Black_Measles),0.931030,0.291185,0.219697,0.000000,0.000000,0.000000,0.000000,0.011264,...,0.833408,0.804557,0.799515,0.828740,0.763529,0.804557,0.757735,0.759904,0.718102,0.729924
4,Grape Esca (Black_Measles) (102).JPG,Grape Esca (Black_Measles),0.956754,0.191401,0.219054,0.000000,0.000000,0.000000,0.000000,0.001587,...,0.860502,0.815168,0.793141,0.845038,0.796258,0.815168,0.744673,0.774935,0.747954,0.736760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1820,grape leaf black rot (95).JPG,grape leaf black rot,0.938239,0.033049,0.344402,0.001689,0.000000,0.000000,0.000000,0.000080,...,0.861067,0.836121,0.814615,0.838965,0.820824,0.836121,0.784267,0.797215,0.794435,0.792455
1821,grape leaf black rot (96).JPG,grape leaf black rot,0.830580,0.556774,0.011387,0.000000,0.000000,0.000000,0.000942,0.003082,...,0.874495,0.835284,0.793708,0.826119,0.816167,0.835284,0.744758,0.759554,0.776001,0.776124
1822,grape leaf black rot (97).JPG,grape leaf black rot,0.931481,0.101385,0.348219,0.028401,0.000000,0.000000,0.000000,0.000602,...,0.872304,0.831270,0.810663,0.852210,0.811875,0.831270,0.768062,0.793662,0.770802,0.760985
1823,grape leaf black rot (98).JPG,grape leaf black rot,0.842217,0.073849,0.534054,0.001950,0.000000,0.000000,0.000000,0.000443,...,0.909724,0.878259,0.828719,0.857464,0.857059,0.878259,0.784617,0.796873,0.822408,0.830377


In [76]:
valid_df.to_csv(f"features/{TARGET_SIZE}/valid_features.csv", index=False)